In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

RUN_MODELS = ["distilbert-base-uncased", "bert-base-uncased"]
RUN_TASKS = ["sst2", "cola", "mrpc"]
SEEDS = [0, 1, 2]
K_FT = [6, 4]                      # retained heads per layer (of 12)
CRITERIA = ["random", "frob", "op", "chain", "vo", "grad", "act"]
FT_N = 5000                        # recovery examples (capped by the training-set size)
FT_EPOCHS = 1
LR, BATCH, WARMUP, WD, CLIP = 2e-5, 32, 0.1, 0.01, 1.0
GRAD_N = 512                       # dev examples for grad / act scoring (same as prune_eval.py)
OUT_DIR = "/kaggle/working/posthoc_ft_results"
REF_GLOBS = ["/kaggle/input/**/P1_extra_all_results.csv", "/kaggle/input/**/P2_all-results.csv", "/kaggle/input/**/P3_all-results.csv"]
SMOKE = False                      # True: 1 checkpoint, K=[4], criteria op+grad, 10 steps (sanity check only)
SEARCH = [
    "/kaggle/input/notebooks/*/*/ckpt/*/done.json",
    "/kaggle/input/*/ckpt/*/done.json",
    "/kaggle/input/**/done.json",
    "/kaggle/working/ckpt/**/done.json",
]
MAX_LEN = 128

import glob, json, time
import numpy as np
import pandas as pd

if SMOKE:
    K_FT, CRITERIA, FT_N, OUT_DIR = [4], ["op", "grad"], 320, OUT_DIR + "_smoke"


def uniform_keep(score, k):
    L, H = score.shape
    keep = np.zeros((L, H), bool)
    for l in range(L):
        keep[l, np.argsort(-score[l], kind="stable")[:k]] = True
    return keep


def vo_norm(Wo_h, Wv_h):
    """||W_V^h W_O^h||_2 from torch-layout slices Wo_h (D, dh), Wv_h (dh, D); thin-QR identity (see prune_eval_vo.py)."""
    _, R1 = np.linalg.qr(Wo_h)
    _, R2 = np.linalg.qr(Wv_h.T)
    return float(np.linalg.norm(R1 @ R2.T, 2))


def pruned_col_masks(keep, dh):
    """Per layer: bool vector over the H*dh input columns of the output projection, True where the head is PRUNED."""
    return [np.repeat(~keep[l], dh) for l in range(keep.shape[0])]


def eta_str(done, total, elapsed):
    if done <= 0:
        return "?"
    rem = elapsed / done * (total - done)
    return f"{rem / 60:.1f} min"


def build_ref_lookup(df):
    """(model, task, seed, method, k) -> primary metric (fraction) from an earlier prune_eval results table (draw 0)."""
    d = df[df["draw"] == 0]
    out = {}
    for r in d.itertuples(index=False):
        p = {"sst2": r.val_acc, "cola": r.val_mcc, "mrpc": r.val_f1}[r.task]
        out[(r.model, r.task, int(r.seed), r.method, int(r.k))] = float(p)
    return out


def selftest():
    rng = np.random.RandomState(0)
    # vo_norm
    for (D, dh) in [(768, 64), (10, 10)]:
        Wo, Wv = rng.randn(D, dh), rng.randn(dh, D)
        assert abs(vo_norm(Wo, Wv) - np.linalg.norm(Wo @ Wv, 2)) < 1e-8 * max(1, np.linalg.norm(Wo @ Wv, 2))
    # masks: pruned columns are exactly those of pruned heads, in head order
    keep = np.array([[True, False, True], [False, False, True]])
    m = pruned_col_masks(keep, 2)
    assert m[0].tolist() == [False, False, True, True, False, False]
    assert m[1].tolist() == [True, True, True, True, False, False]
    assert sum(int(x.sum()) for x in m) == int((~keep).sum()) * 2
    # budget: every criterion keeps exactly k*L heads
    sc = rng.rand(6, 12)
    for k in (6, 4):
        assert uniform_keep(sc, k).sum() == 6 * k
    # random-draw formula identical to prune_eval.py (draw 0): same mask for the same seed/k
    a = uniform_keep(np.random.RandomState(100000 * 3 + 100 * 6 + 0).rand(6, 12), 6)
    b = uniform_keep(np.random.RandomState(100000 * 3 + 100 * 6 + 0).rand(6, 12), 6)
    assert (a == b).all()
    # eta
    assert eta_str(0, 10, 5) == "?" and eta_str(5, 10, 300) == "5.0 min"
    # reference lookup
    df = pd.DataFrame([dict(model="m", task="cola", seed=0, method="asicr_uniform", k=6, draw=0, val_acc=.8, val_mcc=.3, val_f1=np.nan),
                       dict(model="m", task="sst2", seed=1, method="frob_uniform", k=4, draw=1, val_acc=.7, val_mcc=np.nan, val_f1=np.nan),
                       dict(model="m", task="sst2", seed=1, method="frob_uniform", k=4, draw=0, val_acc=.75, val_mcc=np.nan, val_f1=np.nan)])
    lk = build_ref_lookup(df)
    assert lk[("m", "cola", 0, "asicr_uniform", 6)] == .3 and lk[("m", "sst2", 1, "frob_uniform", 4)] == .75
    print("selftest OK (QR shortcut, pruned-column masks, budgets, seeds, eta, reference lookup)")


selftest()


import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import matthews_corrcoef, f1_score

dev_ = torch.device("cuda" if torch.cuda.is_available() else "cpu")
KEYS = {"sst2": ("sentence", None), "cola": ("sentence", None), "mrpc": ("sentence1", "sentence2")}
CRIT_TO_REF = {"frob": "frob_uniform", "op": "asicr_uniform", "chain": "asicr_full_uniform", "grad": "grad_uniform"}


def get_attn(model):
    if hasattr(model, "distilbert"):
        ls = model.distilbert.transformer.layer
        acc = [(l.attention.q_lin, l.attention.k_lin, l.attention.v_lin, l.attention.out_lin) for l in ls]
        return acc, model.config.n_heads, model.config.dim
    ls = model.bert.encoder.layer
    acc = [(l.attention.self.query, l.attention.self.key, l.attention.self.value, l.attention.output.dense) for l in ls]
    return acc, model.config.num_attention_heads, model.config.hidden_size


def prep(task, tok):
    ds = load_dataset("nyu-mll/glue", task)
    k1, k2 = KEYS[task]

    def f(b):
        return tok(b[k1], b[k2], truncation=True, max_length=MAX_LEN) if k2 else tok(b[k1], truncation=True, max_length=MAX_LEN)

    split = ds["train"].train_test_split(test_size=0.05, seed=0)          # identical to fine-tuning script
    tr = split["train"]
    idx = np.random.RandomState(1).choice(len(tr), min(FT_N, len(tr)), replace=False)   # fixed, seed-independent recovery subset
    parts = {"train_ft": tr.select(sorted(idx.tolist())), "dev": split["test"], "val": ds["validation"]}
    cols = ["input_ids", "attention_mask", "label"]
    out = {}
    for n, d in parts.items():
        d = d.map(f, batched=True)
        out[n] = d.remove_columns([c for c in d.column_names if c not in cols])
    out["dev_grad"] = out["dev"].select(range(min(GRAD_N, len(out["dev"]))))
    return out


def make_collate(tok):
    def collate(batch):
        labels = torch.tensor([b["label"] for b in batch])
        enc = tok.pad([{k: b[k] for k in ("input_ids", "attention_mask")} for b in batch], return_tensors="pt")
        enc["labels"] = labels
        return enc
    return collate


@torch.no_grad()
def predict(model, ds, collate, bs=128):
    model.eval()
    lg, lb = [], []
    for b in DataLoader(ds, batch_size=bs, collate_fn=collate):
        lab = b.pop("labels")
        b = {k: v.to(dev_) for k, v in b.items()}
        with torch.autocast("cuda", dtype=torch.float16, enabled=dev_.type == "cuda"):
            lg.append(model(**b).logits.float().cpu())
        lb.append(lab)
    return torch.cat(lg).numpy(), torch.cat(lb).numpy()


def metrics(task, logits, labels):
    p = logits.argmax(1)
    loss = torch.nn.functional.cross_entropy(torch.tensor(logits), torch.tensor(labels)).item()
    m = {"acc": float((p == labels).mean()), "loss": loss}
    m["mcc"] = float(matthews_corrcoef(labels, p)) if task == "cola" else np.nan
    m["f1"] = float(f1_score(labels, p)) if task == "mrpc" else np.nan
    m["primary"] = {"sst2": m["acc"], "cola": m["mcc"], "mrpc": m["f1"]}[task]
    return m


def head_scores(model):
    """frob / op / chain from the DENSE weights (verbatim from prune_eval.py, without kappa)."""
    acc, H, D = get_attn(model)
    dh = D // H
    L = len(acc)
    S = {k: np.zeros((L, H)) for k in ["frob", "op", "chain"]}
    with torch.no_grad():
        for l, (q, k, v, o) in enumerate(acc):
            Wq, Wk, Wv, Wo = [m.weight.detach().float().cpu() for m in (q, k, v, o)]
            for h in range(H):
                sl = slice(h * dh, (h + 1) * dh)
                Woh = Wo[:, sl]
                sv = torch.linalg.svdvals(Woh)
                S["op"][l, h] = sv[0].item()
                S["frob"][l, h] = torch.linalg.matrix_norm(Woh, "fro").item()
                bq = torch.linalg.matrix_norm(Wq[sl, :], 2).item()
                bk = torch.linalg.matrix_norm(Wk[sl, :], 2).item()
                bv = torch.linalg.matrix_norm(Wv[sl, :], 2).item()
                S["chain"][l, h] = bq * bk * bv * sv[0].item()
    return S


def vo_scores(model):
    acc, H, D = get_attn(model)
    dh, L = D // H, len(acc)
    out = np.zeros((L, H))
    with torch.no_grad():
        for l, (_, _, v, o) in enumerate(acc):
            Wv = v.weight.detach().double().cpu().numpy()
            Wo = o.weight.detach().double().cpu().numpy()
            for h in range(H):
                sl = slice(h * dh, (h + 1) * dh)
                out[l, h] = vo_norm(Wo[:, sl], Wv[sl, :])
    return out


def grad_importance(model, ds, collate):
    """Michel et al. (2019)-style: mean |dL/dxi_h|, batch size 1, needs labels (verbatim from prune_eval.py)."""
    acc, H, D = get_attn(model)
    dh, L = D // H, len(acc)
    xi = torch.ones(L, H, device=dev_, requires_grad=True)
    hooks = []
    for l, (_, _, _, o) in enumerate(acc):
        def pre(mod, args, l=l):
            x = args[0]
            B, T, Dm = x.shape
            return ((x.reshape(B, T, H, dh) * xi[l].view(1, 1, H, 1)).reshape(B, T, Dm),)
        hooks.append(o.register_forward_pre_hook(pre))
    imp = torch.zeros(L, H, device=dev_)
    model.eval()
    for b in DataLoader(ds, batch_size=1, collate_fn=collate):
        b = {k: v.to(dev_) for k, v in b.items()}
        loss = model(**b).loss
        g, = torch.autograd.grad(loss, xi)
        imp += g.abs()
    for h_ in hooks:
        h_.remove()
    return (imp / len(ds)).detach().cpu().numpy()


@torch.no_grad()
def act_importance(model, ds, collate, bs=32):
    """Label-free, forward-only: mean over valid tokens of ||W_O,h c_h||_2 (verbatim from grad_samples.py)."""
    acc, H, D = get_attn(model)
    dh, L = D // H, len(acc)
    grams = []
    for (_, _, _, o) in acc:
        Wo = o.weight.detach().float().reshape(D, H, dh)
        grams.append(torch.einsum("ohd,ohe->hde", Wo, Wo))
    sums = torch.zeros(L, H, device=dev_)
    state = {"mask": None, "n": 0}
    hooks = []
    for l, (_, _, _, o) in enumerate(acc):
        def pre(mod, args, l=l):
            x = args[0].float().reshape(args[0].shape[0], args[0].shape[1], H, dh)
            q = torch.einsum("bthd,hde,bthe->bth", x, grams[l], x).clamp_min(0).sqrt()
            sums[l] += (q * state["mask"].unsqueeze(-1)).sum((0, 1))
        hooks.append(o.register_forward_pre_hook(pre))
    model.eval()
    for b in DataLoader(ds, batch_size=bs, collate_fn=collate):
        b.pop("labels")                                                  # labels are NOT used
        b = {k: v.to(dev_) for k, v in b.items()}
        state["mask"] = b["attention_mask"].float()
        state["n"] += int(b["attention_mask"].sum().item())
        with torch.autocast("cuda", dtype=torch.float16, enabled=dev_.type == "cuda"):
            model(**b)
    for h_ in hooks:
        h_.remove()
    return (sums / max(state["n"], 1)).cpu().numpy()


def zero_heads(model, keep):
    acc, H, D = get_attn(model)
    dh = D // H
    with torch.no_grad():
        for l, (_, _, _, o) in enumerate(acc):
            for h in range(H):
                if not keep[l, h]:
                    o.weight[:, h * dh:(h + 1) * dh] = 0.0


def pruned_max_abs(model, keep):
    """Largest |weight| left in the W_O columns of pruned heads (must be exactly 0.0)."""
    acc, H, D = get_attn(model)
    dh = D // H
    mx = 0.0
    with torch.no_grad():
        for l, (_, _, _, o) in enumerate(acc):
            for h in range(H):
                if not keep[l, h]:
                    mx = max(mx, float(o.weight[:, h * dh:(h + 1) * dh].abs().max()))
    return mx


def make_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=dev_.type == "cuda")
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=dev_.type == "cuda")


def finetune(model, train_ds, collate, seed, keep):
    """One fixed recovery schedule. Pruned heads are kept removed (grad zeroed before clipping, columns re-zeroed after each step)."""
    acc, H, D = get_attn(model)
    dh = D // H
    ws = [o.weight for (_, _, _, o) in acc]
    masks = [torch.from_numpy(m).to(dev_) for m in pruned_col_masks(keep, dh)]
    has_p = [bool(m.any()) for m in masks]
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, collate_fn=collate, generator=torch.Generator().manual_seed(seed))
    total = len(loader) * FT_EPOCHS
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP * total), total)
    scaler = make_scaler()
    model.train()
    t0, last = time.time(), float("nan")
    for ep in range(FT_EPOCHS):
        for b in loader:
            b = {k: v.to(dev_) for k, v in b.items()}
            with torch.autocast("cuda", dtype=torch.float16, enabled=dev_.type == "cuda"):
                loss = model(**b).loss
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            for w, m, hp in zip(ws, masks, has_p):
                if hp and w.grad is not None:
                    w.grad[:, m] = 0.0
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
            scaler.step(opt); scaler.update(); sched.step()
            with torch.no_grad():
                for w, m, hp in zip(ws, masks, has_p):
                    if hp:
                        w[:, m] = 0.0
            last = float(loss.item())
    model.eval()
    return time.time() - t0, last


def discover():
    runs = {}
    for pat in SEARCH:
        for p in glob.glob(pat, recursive=True):
            try:
                r = json.load(open(p))
            except Exception:
                continue
            runs[(r["model"], r["task"], r["seed"])] = os.path.dirname(p)
    sel = {k: v for k, v in runs.items() if k[0] in RUN_MODELS and k[1] in RUN_TASKS and k[2] in SEEDS}
    if not sel:
        print("No matching checkpoints. All done.json files seen:", sorted(runs.keys()))
    sel = dict(sorted(sel.items()))
    if SMOKE:
        sel = dict(list(sel.items())[:1])
    return sel


def load_reference():
    frames = []
    for pat in REF_GLOBS:
        for p in glob.glob(pat, recursive=True):
            try:
                frames.append(pd.read_csv(p))
            except Exception:
                pass
    if not frames:
        print("(no earlier result CSVs found -> before-fine-tuning scores are not cross-checked)")
        return {}
    lk = build_ref_lookup(pd.concat(frames))
    print(f"reference lookup: {len(lk)} earlier scores loaded for cross-checking")
    return lk


TIMER = {"t0": time.time(), "done": 0, "total": 0}


def run_checkpoint(key, path, cache, REF):
    mname, task, seed = key
    tag = f"{mname}_{task}_seed{seed}"
    odir = f"{OUT_DIR}/{tag}"
    if os.path.exists(f"{odir}/DONE"):
        print("skip (done):", tag); return
    os.makedirs(odir, exist_ok=True)
    partial = f"{odir}/partial.csv"
    rows = pd.read_csv(partial).to_dict("records") if os.path.exists(partial) else []
    done_keys = {(r["method"], int(r["k"])) for r in rows}
    if (mname, task) not in cache:
        tok = AutoTokenizer.from_pretrained(path)
        cache[(mname, task)] = (tok, prep(task, tok), make_collate(tok))
    tok, data, collate = cache[(mname, task)]

    base = AutoModelForSequenceClassification.from_pretrained(path).to(dev_).eval()
    acc, H, D = get_attn(base)
    L = len(acc)
    S = head_scores(base)
    VO = vo_scores(base)
    G = grad_importance(base, data["dev_grad"], collate)
    A = act_importance(base, data["dev_grad"], collate)
    del base
    torch.cuda.empty_cache()
    scores = {"frob": S["frob"], "op": S["op"], "chain": S["chain"], "vo": VO, "grad": G, "act": A}
    print(f"\n=== {tag}: L={L} H={H}  recovery set={len(data['train_ft'])} examples, {FT_EPOCHS} epoch(s)")

    plan = [("dense", H)] + [(c, k) for k in K_FT for c in CRITERIA]
    for crit, k in plan:
        if (crit, k) in done_keys:
            continue
        if crit == "dense":
            keep = np.ones((L, H), bool)
        elif crit == "random":
            keep = uniform_keep(np.random.RandomState(100000 * seed + 100 * k + 0).rand(L, H), k)     # = draw 0 of prune_eval.py
        else:
            keep = uniform_keep(scores[crit], k)
        assert keep.sum() == k * L, (crit, k, keep.sum())

        model = AutoModelForSequenceClassification.from_pretrained(path).to(dev_).eval()
        zero_heads(model, keep)
        lv, yv = predict(model, data["val"], collate)
        pre = metrics(task, lv, yv)
        secs, last_loss = finetune(model, data["train_ft"], collate, seed, keep)
        lv, yv = predict(model, data["val"], collate)
        post = metrics(task, lv, yv)
        resid = pruned_max_abs(model, keep)
        assert resid == 0.0, f"pruned head came back to life (max |w| = {resid})"
        ref_key = (mname, task, seed, CRIT_TO_REF.get(crit, "?"), k)
        ref_diff = abs(pre["primary"] - REF[ref_key]) if ref_key in REF else float("nan")
        rows.append(dict(model=mname, task=task, seed=seed, method=crit, k=k, n_kept=int(keep.sum()),
                         ft_n=len(data["train_ft"]), ft_epochs=FT_EPOCHS, lr=LR,
                         pre_acc=pre["acc"], pre_mcc=pre["mcc"], pre_f1=pre["f1"], pre_loss=pre["loss"], pre_primary=pre["primary"],
                         post_acc=post["acc"], post_mcc=post["mcc"], post_f1=post["f1"], post_loss=post["loss"], post_primary=post["primary"],
                         ft_seconds=secs, last_train_loss=last_loss, pruned_weights_max_abs=resid, ref_diff=ref_diff))
        pd.DataFrame(rows).to_csv(partial, index=False)
        TIMER["done"] += 1
        flag = "" if not (ref_diff == ref_diff) else ("  [ref OK]" if ref_diff < 0.005 else f"  !! REF MISMATCH {ref_diff:.4f} (different setup/checkpoint?)")
        print(f"  [{TIMER['done']}/{TIMER['total']}] {crit:6s} k={k:2d}: {100 * pre['primary']:6.2f} -> {100 * post['primary']:6.2f}   "
              f"({secs:.0f}s train){flag}   ETA {eta_str(TIMER['done'], TIMER['total'], time.time() - TIMER['t0'])}")
        del model
        torch.cuda.empty_cache()

    df = pd.DataFrame(rows)
    df.to_csv(f"{odir}/results.csv", index=False)
    dense_row = df[df.method == "dense"]
    try:
        stored = json.load(open(f"{path}/done.json"))["val"]["acc"]
        print(f"  dense check vs done.json: |diff| = {abs(float(dense_row.pre_acc.iloc[0]) - stored):.4f}")
    except Exception:
        pass
    open(f"{odir}/DONE", "w").write("ok")


def main():
    runs = discover()
    print(f"Found {len(runs)} checkpoints:")
    for k, v in runs.items():
        print("  ", k, "->", v)
    assert runs, "No checkpoints found: add the fine-tuning notebook outputs via Add Input -> Notebook Output Files"
    TIMER["total"] = len(runs) * (1 + len(K_FT) * len(CRITERIA))
    REF = load_reference()
    cache = {}
    for key, path in runs.items():
        run_checkpoint(key, path, cache, REF)
    allr = [pd.read_csv(p) for p in glob.glob(f"{OUT_DIR}/*/results.csv")]
    if allr:
        A = pd.concat(allr)
        A.to_csv(f"{OUT_DIR}/ALL_posthoc_results.csv", index=False)
        A["pre"], A["post"] = 100 * A.pre_primary, 100 * A.post_primary
        s = A.groupby(["model", "task", "k", "method"])[["pre", "post"]].mean().round(1)
        print("\nMean score x100 before -> after recovery:")
        print(s.to_string())
    print("\nALL DONE ->", OUT_DIR)


if __name__ == "__main__" or True:
    main()

selftest OK (QR shortcut, pruned-column masks, budgets, seeds, eta, reference lookup)
Found 12 checkpoints:
   ('bert-base-uncased', 'sst2', 0) -> /kaggle/input/notebooks/krishnaspathak/scc-notebook3/ckpt/bert-base-uncased_sst2_seed0
   ('bert-base-uncased', 'sst2', 1) -> /kaggle/input/notebooks/krishnaspathak/scc-notebook3/ckpt/bert-base-uncased_sst2_seed1
   ('bert-base-uncased', 'sst2', 2) -> /kaggle/input/notebooks/krishnaspathak/scc-notebook3/ckpt/bert-base-uncased_sst2_seed2
   ('distilbert-base-uncased', 'cola', 0) -> /kaggle/input/notebooks/krishnaspathak/scc-notebook2/ckpt/distilbert-base-uncased_cola_seed0
   ('distilbert-base-uncased', 'cola', 1) -> /kaggle/input/notebooks/krishnaspathak/scc-notebook2/ckpt/distilbert-base-uncased_cola_seed1
   ('distilbert-base-uncased', 'cola', 2) -> /kaggle/input/notebooks/krishnaspathak/scc-notebook2/ckpt/distilbert-base-uncased_cola_seed2
   ('distilbert-base-uncased', 'mrpc', 0) -> /kaggle/input/notebooks/krishnaspathak/scc-notebook2/ck

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

sst2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.11MB            

sst2/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sst2/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 72.8kB            

sst2/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (448 outputs left out)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

  [179/180] grad   k= 4:  87.61 ->  89.79   (9s train)   ETA 0.2 min


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

  [180/180] act    k= 4:  87.61 ->  89.22   (9s train)   ETA 0.0 min
  dense check vs done.json: |diff| = 0.0000

Mean score x100 before -> after recovery:
                                         pre  post
model                   task k  method            
bert-base-uncased       sst2 4  act     89.6  90.9
                                chain   80.0  86.6
                                frob    78.1  87.8
                                grad    89.1  90.7
                                op      79.0  86.6
                                random  78.2  88.5
                                vo      79.5  85.7
                             6  act     91.3  91.8
                                chain   88.1  90.2
                                frob    88.4  90.4
                                grad    91.4  91.8
                                op      88.7  90.8
                                random  89.4  91.0
                                vo      84.9  87.2
                            